# TUT1. Agentive Data Scraping

In [17]:
!pip install pkgman
from pkgman import include
include("pandas", "openai", "tqdm", "requests", "rich", "requests", "re")
from rich import print as pp

[pkgman] 7 packages have been imported.


In [ ]:
# The following key is expring today; for tuturial only

# We first need to setup the API key for using OpenAI's services. In this tutorial, we use online service for convenience
OPENAI_API_KEY = "PASTE_YOUR_KEY_HERE"

NEWS_URL = "https://www.nbcnews.com/world"


## 1. Collaborate with GenAI

In this first section, we are going to be collaborative with AI models -- try to assign a task to AI first:

## 1.1 Prompting your Gemini

For simplicity, we are gonna use Jina for simplying the problem. Jina is a new company providing parsed web contents in markdown language, which is super helpful for us.

```
Write a neat function to retrieve data using Jina AI's service. 

- Data source: `https://www.nbcnews.com/world`
- Example requests: `curl "https://r.jina.ai/https://www.example.com"`
- Input: URL of the webpage to retrieve
- Output: Parse the markdown output to extract link titles and urls; filter titles with more than four english words (not characters); return a list of dicts

Run and print the last five items.
```

## 1.2 Executing the generative scraper

What if you can not fully understand the code?   
Ask to explain (concisely, in detail, or in any popular language)!

In [19]:
def retrieve_and_parse_jina_ai(url):
    """
    Retrieves and parses content from a given URL using Jina AI.

    Args:
        url (str): The URL of the content to retrieve.

    Returns:
        list: A list of dictionaries, each containing 'title' and 'url' for the extracted links.
              Returns an empty list if content retrieval fails or no links are found.
    """
    jina_ai_url = f"https://r.jina.ai/{url}"
    extracted_links = []
    filtered_links = [] # Initialize filtered_links
    try:
        response = requests.get(jina_ai_url, headers={'Accept': 'application/json'})
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        markdown_content = response.text

        # Use regex to find all markdown links [title](url)
        # Group 1 captures the title, Group 2 captures the URL
        links_found = re.findall(r'\[(.*?)\]\((.*?)\)', markdown_content)

        for title, link_url in links_found:
            if title and link_url:
                extracted_links.append({'title': title, 'url': link_url})
        
        # Filter links based on title word count
        for link in extracted_links:
            title = link['title']
            # Clean the title: remove punctuation and convert to lowercase
            cleaned_title = re.sub(r'[^a-zA-Z\s]', '', title).lower()
            # Split into words and count, filtering out empty strings from split()
            words = [word for word in cleaned_title.split() if word]
            
            if len(words) > 4:
                filtered_links.append(link)

        return filtered_links # Return filtered_links
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving content from Jina AI: {e}")
        return []

print("Function 'retrieve_and_parse_jina_ai' updated successfully with markdown parsing and word count filtering.")

Function 'retrieve_and_parse_jina_ai' updated successfully with markdown parsing and word count filtering.


In [20]:
pp(retrieve_and_parse_jina_ai(NEWS_URL)[:3])

[
    {
        'title': 'Early Amazon Big Spring Sale Deals',
        'url': 'https://www.nbcnews.com/select/shopping/amazon-big-spring-sale-early-deals-2026-rcna263339'
    },
    {
        'title': '![Image 1: 1950s vintage cars remain a daily transportation in Cuba',
        'url': 
'https://media-cldnry.s-nbcnews.com/image/upload/t_focal-762x508,f_auto,q_auto:best/rockcms/2026-03/260316-cuba-mb-
1251-deb989.jpg'
    },
    {
        'title': 'Cuba will allow nationals abroad to invest in and own businesses on the island, economic czar 
says',
        'url': 
'https://www.nbcnews.com/world/cuba/cuba-allow-nationals-living-abroad-invest-businesses-island-economy-rcna263637'
    }
]

### 1.3 Debugging with GenAI

Sometimes, codes are written with bugs. Try to ask Gemini to fix the bug within the following edited version of the code.

By prompting: `fix: ` and the error logs you encountered.

In [22]:
## Error left on purpose for testing error handling

def retrieve_and_parse_jina_ai(url):
    """
    Retrieves and parses content from a given URL using Jina AI.

    Args:
        url (str): The URL of the content to retrieve.

    Returns:
        list: A list of dictionaries, each containing 'title' and 'url' for the extracted links.
              Returns an empty list if content retrieval fails or no links are found.
    """
    jina_ai_url = f"https://r.jina.ai/{url}"
    extracted_links = []
    filtered_links = [] # Initialize filtered_links
    try:
        response = requests.get(jina_ai_url, headers={'Accept': 'application/json'})
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
        markdown_content = response.text

        # Use regex to find all markdown links [title](url)
        # Group 1 captures the title, Group 2 captures the URL
        llnks_found = re.findall(r'\[(.*?)\]\((.*?)\)', markdown_content)

        for title, link_url in links_found:
            if title and link_url:
                extracted_links.append({'title': title, 'url': link_url})
        
        # Filter links based on title word count
        for link in extracted_links:
            title = link['title']
            # Clean the title: remove punctuation and convert to lowercase
            cleaned_title = re.sub(r'[^a-zA-Z\s]', '', title).lower()
            # Split into words and count, filtering out empty strings from split()
            words = [word for word in cleaned_title.split() if word]
            
            if len(words) > 4:
                filtered_links.append(link)

        return filtered_links # Return filtered_links
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving content from Jina AI: {e}")
        return []

retrieve_and_parse_jina_ai(NEWS_URL)

NameError: name 'links_found' is not defined

Once the bug was resolvedd, accept the edit and run again. Now everything should be fine.

## 2. Scraping pipeline with the help of GenAI

In previous section, we have introduced the core function in scraping. However, a complete scraping pipeline may also involve storing the scraped data, handling errors, and do some post-processing.

You can try to ask Gemini to write a complete scraping pipeline for you, and test the error handling by using the above function with a wrong URL.

## 3. Advanced Data Scraping

Some problems may arise if we want to have some sensitive data (e.g., X.com protects its content very well and X is an important social media).

**Solutions:**
- Official API: https://github.com/daijro/camoufox
- Apify (https://apify.com/), Twitterapi (https://twitterapi.io/), and other 3-rd party scraping services
- A well-designed opensource project: [https://github.com/the-convocation/twitter-scraper](https://github.com/the-convocation/twitter-scraper) which handles everything quite well (verified yesterday; 18 March, 2026)
- Collaborate with AI to supercharge your own crawler with un-detectable methods
  - https://github.com/ultrafunkamsterdam/nodriver
  - https://github.com/daijro/camoufox

However, this is far beyond today's discussion and the use may be unethical so I will leave this map here in case you have these problems.